# 推荐系统工程：隐式反馈、两阶段召回与排序

推荐系统不是一次 `model.predict(user, item)`。本 Notebook 用受控交互日志实现：全局时间切分、隐式反馈、热门/内容/协同多路召回、BPR pairwise 学习、候选融合、排序、Recall/MRR/nDCG、冷启动、库存/ACL 过滤、曝光偏差与版本化服务。

## 学习目标

1. 区分曝光、点击、加购、购买与真正负反馈；
2. 避免随机切分把未来行为泄漏给过去；
3. 从零实现 item-item 召回和 BPR 更新；
4. 分开评估 candidate recall 与 final ranking；
5. 处理已看去重、冷启动、业务过滤、反馈回路和可回滚发布。

> 合成用户偏好高度规则化，离线结果只用于验证链路。真实推荐受曝光策略、位置偏差、延迟反馈和用户兴趣漂移影响，必须经过线上实验。

## 1. 业务目标与事件合同

目标先明确：本例做“未来两天可能交互的 top-N item”，不是评分预测。事件至少有 `event_id、user_id、item_id、event_time、event_type、request_id、position、was_exposed、tenant`；item 有稳定 ID、类目、上架状态、库存、质量与版本。

点击是有偏的隐式信号：未点击可能是没兴趣，也可能根本没曝光。训练、评估与日志都不能把所有未观察 item 直接写成真负例。删除用户数据时还要处理聚合共现、embedding、特征库和训练快照。

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
import hashlib
import json
import math
import numpy as np

RNG = np.random.default_rng(23)
N_USERS, N_ITEMS, N_CATEGORIES = 8, 24, 4

@dataclass(frozen=True)
class Item:
    item_id: int
    category: int
    tenant: str
    active: bool
    in_stock: bool
    quality: float

@dataclass(frozen=True)
class Event:
    event_id: str
    user_id: int
    item_id: int
    round_id: int
    event_time: datetime
    event_type: str
    tenant: str

items = [Item(i, i // 6, "tenant-a", True, True, 0.6 + 0.03 * (i % 6)) for i in range(N_ITEMS)]
secret_item = Item(900, 0, "tenant-b", True, True, 1.0)
preferences = {user: (user % 4, (user + 1) % 4) for user in range(N_USERS)}
event_types = ("click", "cart", "purchase")
start = datetime(2026, 1, 1, tzinfo=timezone.utc)
events = []
for round_id in range(10):
    for user in range(N_USERS):
        category = preferences[user][round_id % 2]
        item_id = category * 6 + ((user + round_id // 2) % 6)
        events.append(Event(
            f"evt-{round_id:02d}-{user:02d}", user, item_id, round_id,
            start + timedelta(days=round_id, minutes=user),
            event_types[(user + round_id) % len(event_types)], "tenant-a",
        ))

assert len({event.event_id for event in events}) == len(events) == 80
assert all(event.item_id < N_ITEMS for event in events)
print("events=", len(events), "users=", N_USERS, "items=", N_ITEMS)


## 2. 隐式反馈与全局时间切分

`purchase > cart > click` 可用于置信度或采样权重，但不意味着一次购买等于五个独立正例。去重窗口、机器人过滤、退款/取消和迟到事件要写进标签规则版本。

本例按全局 round 切：0–5 train、6–7 validation、8–9 test。validation 用于预算/权重，test 冻结后只报一次。用户同时出现在各 split 是“已有用户未来推荐”的任务合同，不是实体泄漏；若目标是新用户冷启动，必须另做 user-disjoint 测试。

In [ ]:
train_events = [event for event in events if event.round_id <= 5]
valid_events = [event for event in events if 6 <= event.round_id <= 7]
test_events = [event for event in events if event.round_id >= 8]

def user_items(rows):
    result = defaultdict(set)
    for event in rows:
        result[event.user_id].add(event.item_id)
    return {user: set(result[user]) for user in range(N_USERS)}

train_positive = user_items(train_events)
valid_positive = user_items(valid_events)
test_positive = user_items(test_events)
for user in range(N_USERS):
    assert train_positive[user].isdisjoint(valid_positive[user] | test_positive[user])
    assert valid_positive[user].isdisjoint(test_positive[user])
assert max(e.event_time for e in train_events) < min(e.event_time for e in valid_events)
assert max(e.event_time for e in valid_events) < min(e.event_time for e in test_events)
print({"train": len(train_events), "validation": len(valid_events), "test": len(test_events)})


## 3. 两阶段架构：先保 Recall，再精排

召回层可以并行使用热门、item-item 协同、内容/类目、向量和规则通道，每路有独立预算；融合后去重，再交给较贵 ranker。召回层漏掉的 item，排序模型无法补回。

热门适合冷启动但容易头部化；内容通道可服务新 item；协同通道捕捉共同行为但对稀疏用户差。所有通道都必须先遵守 tenant、上架、库存和合规过滤。

In [ ]:
EVENT_WEIGHT = {"click": 1.0, "cart": 2.0, "purchase": 4.0}
popularity = np.zeros(N_ITEMS)
for event in train_events:
    popularity[event.item_id] += EVENT_WEIGHT[event.event_type]

def popular_candidates(seen: set[int], limit: int) -> list[int]:
    candidates = [item.item_id for item in items if item.item_id not in seen and item.active and item.in_stock]
    return sorted(candidates, key=lambda item_id: (-popularity[item_id], item_id))[:limit]

def content_candidates(user: int, seen: set[int], limit: int) -> list[int]:
    affinity = Counter(items[item_id].category for item_id in train_positive[user])
    candidates = [item.item_id for item in items if item.item_id not in seen and item.active and item.in_stock]
    return sorted(candidates, key=lambda item_id: (-affinity[items[item_id].category], -items[item_id].quality, item_id))[:limit]

assert len(popular_candidates(set(), 5)) == 5
assert all(item_id not in train_positive[0] for item_id in content_candidates(0, train_positive[0], 10))


## 4. Item-item 协同召回

把每个 item 表示成训练期交互用户集合，Jaccard 为 $|U_i\cap U_j|/|U_i\cup U_j|$。查询时从用户近期 item 的邻居扩展。生产实现通常使用时间衰减、行为权重、共现去热门修正和离线 top-neighbor 表；不能把 validation/test 行为写进共现图。

In [ ]:
item_users = {item_id: set() for item_id in range(N_ITEMS)}
for event in train_events:
    item_users[event.item_id].add(event.user_id)

def jaccard_items(left: int, right: int) -> float:
    union = item_users[left] | item_users[right]
    return len(item_users[left] & item_users[right]) / len(union) if union else 0.0

def item_item_candidates(user: int, seen: set[int], limit: int) -> list[int]:
    scores = defaultdict(float)
    for source in train_positive[user]:
        for target in range(N_ITEMS):
            if target not in seen and target != source:
                scores[target] = max(scores[target], jaccard_items(source, target))
    return sorted(scores, key=lambda target: (-scores[target], target))[:limit]

assert item_users[0]
assert math.isclose(jaccard_items(0, 0), 1.0)
assert all(item_id not in train_positive[1] for item_id in item_item_candidates(1, train_positive[1], 8))


## 5. BPR：优化相对顺序，不是评分回归

对用户 $u$、正例 $i$ 和采样未观察 item $j$，BPR 最大化 $\log\sigma(\hat x_{ui}-\hat x_{uj})$。本例手写矩阵分解 SGD。未观察不等于真负例；采样分布会改变训练目标，未来正例被采成负例是离线诊断重点。

In [ ]:
FACTOR_DIM = 8
user_factor = RNG.normal(0, 0.1, (N_USERS, FACTOR_DIM))
item_factor = RNG.normal(0, 0.1, (N_ITEMS, FACTOR_DIM))
item_bias = np.zeros(N_ITEMS)
negative_audit = []
for epoch in range(500):
    order = RNG.permutation(len(train_events))
    for event_index in order:
        event = train_events[event_index]
        user, positive = event.user_id, event.item_id
        pool = [item_id for item_id in range(N_ITEMS) if item_id not in train_positive[user]]
        negative = int(RNG.choice(pool))
        if epoch == 0:
            negative_audit.append((user, negative))
        margin = user_factor[user] @ (item_factor[positive] - item_factor[negative]) + item_bias[positive] - item_bias[negative]
        gradient = 1 / (1 + np.exp(np.clip(margin, -30, 30)))
        old_user = user_factor[user].copy()
        old_positive = item_factor[positive].copy()
        old_negative = item_factor[negative].copy()
        lr, reg = 0.03, 0.002
        user_factor[user] += lr * (gradient * (old_positive - old_negative) - reg * old_user)
        item_factor[positive] += lr * (gradient * old_user - reg * old_positive)
        item_factor[negative] += lr * (-gradient * old_user - reg * old_negative)
        item_bias[positive] += lr * (gradient - reg * item_bias[positive])
        item_bias[negative] += lr * (-gradient - reg * item_bias[negative])

def bpr_score(user: int, item_id: int) -> float:
    return float(user_factor[user] @ item_factor[item_id] + item_bias[item_id])

assert np.isfinite(user_factor).all() and np.isfinite(item_factor).all()
print("BPR factors trained:", user_factor.shape, item_factor.shape)


## 6. Negative sampling 与 false negative 审计

只从未观察集合采样是常见弱监督假设，但其中可能包含用户未来会喜欢的 item。离线可以用 held-out 正例统计碰撞率来理解风险，不能把 held-out 标签用于生产训练采样。更可靠的负例来自明确曝光后跳过、dislike/退订等信号，并要校正位置和曝光策略偏差。

In [ ]:
future_positive = {user: valid_positive[user] | test_positive[user] for user in range(N_USERS)}
false_negative_count = sum(negative in future_positive[user] for user, negative in negative_audit)
false_negative_rate = false_negative_count / len(negative_audit)
print({"epoch0_samples": len(negative_audit), "future_positive_collision_rate": round(false_negative_rate, 3)})
assert 0.0 <= false_negative_rate <= 1.0


## 7. 多路候选融合与轻量排序

候选记录要保留 channel、原始 rank/score 和召回版本。这里取热门、内容和 item-item 的 union，再用 BPR、类目亲和、热门度与 item 质量组成可解释 rank score。生产 ranker 可换成 GBDT/DNN/序列模型，但训练样本必须模拟真实候选分布，不能在全库随机负例上训练后直接上线。

In [ ]:
def generate_candidates(user: int, seen: set[int], per_channel=10, budget=20):
    channels = {
        "popular": popular_candidates(seen, per_channel),
        "content": content_candidates(user, seen, per_channel),
        "item_item": item_item_candidates(user, seen, per_channel),
    }
    membership = defaultdict(list)
    for channel, ranking in channels.items():
        for item_id in ranking:
            membership[item_id].append(channel)
    candidates = sorted(membership, key=lambda item_id: (-len(membership[item_id]), item_id))[:budget]
    return candidates, {item_id: tuple(membership[item_id]) for item_id in candidates}

def rank_candidates(user: int, candidates: list[int]) -> list[tuple[int, float]]:
    affinity = Counter(items[item_id].category for item_id in train_positive[user])
    scored = []
    for item_id in candidates:
        score = (bpr_score(user, item_id) + 0.5 * affinity[items[item_id].category]
                 + 0.05 * np.log1p(popularity[item_id]) + 0.15 * items[item_id].quality)
        scored.append((item_id, float(score)))
    return sorted(scored, key=lambda row: (-row[1], row[0]))

history_before_test = {user: train_positive[user] | valid_positive[user] for user in range(N_USERS)}
example_candidates, example_channels = generate_candidates(0, history_before_test[0])
example_ranked = rank_candidates(0, example_candidates)
assert all(item_id not in history_before_test[0] for item_id, _ in example_ranked)
print(example_ranked[:5], example_channels)


## 8. 离线评估：全候选、按用户聚合、分阶段看

Candidate Recall@K 判断 gold 是否进入候选；MRR 看第一个 gold；nDCG 支持多个相关 item 的位置折扣。评估必须说明是全 item universe 还是 sampled negatives，后者通常更容易。还要报告 coverage、长尾、新用户/新 item、类目和活跃度分桶。离线日志由旧策略产生，无法无偏回答新策略线上效果。

In [ ]:
def recall_at_k(ranked: list[int], relevant: set[int], k: int) -> float:
    return len(set(ranked[:k]) & relevant) / len(relevant) if relevant else 0.0

def reciprocal_rank(ranked: list[int], relevant: set[int]) -> float:
    return next((1 / rank for rank, item_id in enumerate(ranked, 1) if item_id in relevant), 0.0)

def ndcg_at_k(ranked: list[int], relevant: set[int], k: int) -> float:
    dcg = sum(1 / math.log2(rank + 1) for rank, item_id in enumerate(ranked[:k], 1) if item_id in relevant)
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(k, len(relevant)) + 1))
    return dcg / ideal if ideal else 0.0

rows, all_recommended = [], set()
for user in range(N_USERS):
    candidates, _ = generate_candidates(user, history_before_test[user], per_channel=12, budget=20)
    ranked = [item_id for item_id, _ in rank_candidates(user, candidates)]
    gold = test_positive[user]
    all_recommended.update(ranked[:10])
    rows.append((recall_at_k(ranked, gold, 10), reciprocal_rank(ranked, gold), ndcg_at_k(ranked, gold, 10)))
metrics = np.mean(rows, axis=0)
print({"Recall@10": metrics[0], "MRR": metrics[1], "nDCG@10": metrics[2], "catalog_coverage@10": len(all_recommended)/N_ITEMS})
assert metrics[0] == 1.0
assert metrics[1] > 0 and metrics[2] > 0


## 9. 冷启动、业务过滤与在线合同

新用户没有协同向量，可用授权后的热门、上下文、显式兴趣或探索；新 item 可用内容特征和受控流量。过滤必须在候选层执行：tenant、年龄/地区合规、下架、库存、已购买和 blocklist。排序后再过滤会浪费 top-N 并可能泄漏。

服务输入来自可信 AuthContext，并把请求的 user_id 绑定到认证 subject，避免 IDOR 读取他人画像；后台代查需要单独的 service role 和审计。输出携带 item_id、score/rank、reason/channel、model/index version 和 request_id；缓存键包含用户状态/权限版本与 catalog generation。

In [ ]:
_AUTH_MARKER = object()

@dataclass(frozen=True)
class AuthContext:
    tenant: str
    subject: str
    _marker: object = field(repr=False, compare=False)

def authenticate_demo(token: str):
    if token != "signed-user-0":
        raise PermissionError("认证失败")
    return AuthContext("tenant-a", "user-0", _AUTH_MARKER)

catalog_by_id = {item.item_id: item for item in [*items, secret_item]}

def hard_filter_candidates(candidate_ids: list[int], auth: AuthContext) -> list[int]:
    """在进入排序前执行租户、上架和库存硬过滤，并拒绝未知 item。"""
    return [
        item_id for item_id in candidate_ids
        if item_id in catalog_by_id
        and catalog_by_id[item_id].tenant == auth.tenant
        and catalog_by_id[item_id].active
        and catalog_by_id[item_id].in_stock
    ]

def recommend(user: int | None, auth: AuthContext, top_k=5):
    if not isinstance(auth, AuthContext) or auth._marker is not _AUTH_MARKER or auth.tenant != "tenant-a":
        raise PermissionError("不可信身份或 tenant")
    if not 1 <= top_k <= 10:
        raise ValueError("top_k 必须在 1..10")
    if user is not None and (not 0 <= user < N_USERS or auth.subject != f"user-{user}"):
        raise PermissionError("只能请求当前认证主体的个性化结果")
    if user is None:
        candidates = hard_filter_candidates(popular_candidates(set(), top_k), auth)
        ranked = [(item_id, float(popularity[item_id])) for item_id in candidates]
        mode = "cold_start_popular"
    else:
        seen = history_before_test[user]
        candidates, _ = generate_candidates(user, seen, per_channel=12, budget=20)
        candidates = hard_filter_candidates(candidates, auth)
        ranked = rank_candidates(user, candidates)[:top_k]
        mode = "personalized_two_stage"
    assert all(catalog_by_id[item_id].tenant == auth.tenant and catalog_by_id[item_id].active and catalog_by_id[item_id].in_stock for item_id, _ in ranked)
    return ranked, {"mode": mode, "degraded": False, "catalog_generation": "catalog-g1"}

AUTH = authenticate_demo("signed-user-0")
served, serving_trace = recommend(0, AUTH)
cold, cold_trace = recommend(None, AUTH)
assert served and cold and serving_trace["mode"] != cold_trace["mode"]


## 10. 曝光偏差、反馈回路与安全

历史点击只覆盖旧策略曝光过的 item，热门模型会继续制造热门数据。离线指标不能替代随机探索或 A/B；可记录 propensity 并研究 IPS/DR，但估计稳定性取决于正确的曝光概率。监控类目/创作者覆盖、头部集中度、新颖性、投诉、长期留存与群体差异。

推荐理由不能泄露敏感推断；用户画像、embedding 和相似用户关系属于个人数据。内容安全、年龄限制和法律过滤是硬约束，不能只当 rank feature。

In [ ]:
def matrix_hash(matrix: np.ndarray) -> str:
    return hashlib.sha256(np.ascontiguousarray(matrix).tobytes()).hexdigest()

manifest = {
    "model_version": "bpr-hybrid-v1",
    "train_cutoff": max(e.event_time for e in train_events).isoformat(),
    "event_weight_version": "implicit-weight-v1",
    "candidate_version": "popular-content-itemitem-v1",
    "rank_feature_version": "hybrid-score-v1",
    "user_factor_hash": matrix_hash(user_factor),
    "item_factor_hash": matrix_hash(item_factor),
    "catalog_generation": "catalog-g1",
}
manifest["bundle_sha256"] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
assert len(manifest["bundle_sha256"]) == 64
print(json.dumps(manifest, ensure_ascii=False, indent=2))


## 11. 回归与发布门禁

测试至少覆盖时间边界、event 幂等、已看去重、candidate budget、future feature 禁止、BPR 有限值、gold candidate recall、冷启动、下架/库存、tenant、伪造身份、缓存/manifest。发布时新旧模型影子运行，比较候选交集、分数漂移、延迟和安全过滤，再灰度并可回滚。

In [ ]:
assert len(train_events) == 48 and len(valid_events) == len(test_events) == 16
assert all(event.round_id <= 5 for event in train_events)
assert all(item_id not in history_before_test[0] for item_id, _ in served)
assert len(served) <= 5 and len(cold) <= 5
assert len({item_id for item_id, _ in served}) == len(served)
assert hard_filter_candidates([secret_item.item_id, 0, -1], AUTH) == [0]
assert secret_item.item_id not in {item_id for item_id, _ in served}
assert np.isfinite(metrics).all()
assert manifest["train_cutoff"].startswith("2026-01")
for user in range(N_USERS):
    candidates, _ = generate_candidates(user, history_before_test[user], budget=20)
    assert test_positive[user] <= set(candidates)
try:
    recommend(0, AuthContext("tenant-a", "attacker", object()))
    raise AssertionError("伪造身份必须拒绝")
except PermissionError:
    pass
try:
    recommend(1, AUTH)
    raise AssertionError("不能读取其他用户的个性化结果")
except PermissionError:
    pass
try:
    recommend(0, AUTH, top_k=0)
    raise AssertionError("非法 top_k 必须拒绝")
except ValueError:
    pass
print("推荐时间切分、召回、BPR、排序、冷启动、ACL 与版本断言全部通过。")


## 12. 研究依据与教学边界

- Hu, Koren & Volinsky, [Collaborative Filtering for Implicit Feedback Datasets](https://doi.org/10.1109/ICDM.2008.22), 2008。
- Rendle et al., [BPR: Bayesian Personalized Ranking from Implicit Feedback](https://www.cs.mcgill.ca/~uai2009/papers/UAI2009_0139_48141db02b9f0b02bc7158819ebfa2c7.pdf), UAI 2009。
- Ji et al., [A Critical Study on Data Leakage in Recommender System Offline Evaluation](https://arxiv.org/abs/2010.11060)。
- Zhao et al., [RecBole: Towards a Unified, Comprehensive and Efficient Framework for Recommendation Algorithms](https://arxiv.org/abs/2011.01731)：可用于对照成熟训练/评测框架。

本例没有真实曝光日志、序列模型、特征平台、分布式 ANN、实时状态或反事实估计；BPR 与混合分数只展示职责边界。线上收益必须通过安全灰度实验判断。